In [3]:
import json
import os
import re
from pathlib import Path

import dotenv
from neo4j import GraphDatabase

dotenv.load_dotenv("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")

INPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_contract_kg_normalized.json")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


In [4]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

kg = data["knowledge_graph"]
entities = kg["entities"]
relations = kg.get("relations", [])

print("Entities:", len(entities))
print("Relations:", len(relations))

Entities: 1330
Relations: 2164


In [5]:
ALLOWED_LABELS = {
    "Clause",
    "DefinedTerm",
    "Party",
    "Obligation",
    "Right",
    "Permission",
    "Prohibition",
    "Condition",
    "Reference",
    "Value",
}

def safe_label(label):
    if label not in ALLOWED_LABELS:
        raise ValueError(f"Invalid label: {label}")
    return label


def normalize_text(value):
    if value is None:
        return None
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s_:-]", "", value)
    value = re.sub(r"\s+", "_", value)
    return value


def clean_props(props):
    clean = {}

    for k, v in props.items():
        if v is None:
            continue

        if isinstance(v, (str, int, float, bool)):
            clean[k] = v

        elif isinstance(v, list):
            clean[k] = [
                str(x) for x in v
                if x is not None
            ]

        else:
            clean[k] = json.dumps(v, ensure_ascii=False)

    return clean

In [6]:
# CLEAR_DATABASE = True

# def clear_database(tx):
#     tx.run("MATCH (n) DETACH DELETE n")

# if CLEAR_DATABASE:
#     with driver.session() as session:
#         session.execute_write(clear_database)

#     print("Database cleared.")

In [7]:
def create_constraints(tx):
    constraints = [
        "CREATE CONSTRAINT clause_id IF NOT EXISTS FOR (n:Clause) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT definedterm_id IF NOT EXISTS FOR (n:DefinedTerm) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT party_id IF NOT EXISTS FOR (n:Party) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT obligation_id IF NOT EXISTS FOR (n:Obligation) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT right_id IF NOT EXISTS FOR (n:Right) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT permission_id IF NOT EXISTS FOR (n:Permission) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT prohibition_id IF NOT EXISTS FOR (n:Prohibition) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT condition_id IF NOT EXISTS FOR (n:Condition) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT reference_id IF NOT EXISTS FOR (n:Reference) REQUIRE n.id IS UNIQUE",
        "CREATE CONSTRAINT value_id IF NOT EXISTS FOR (n:Value) REQUIRE n.id IS UNIQUE",
    ]

    for query in constraints:
        tx.run(query)

with driver.session() as session:
    session.execute_write(create_constraints)

print("Constraints created.")

ClientError: {neo4j_code: Neo.ClientError.Schema.IndexAlreadyExists} {message: There already exists an index (:Clause {id}). A constraint cannot be created until the index has been dropped.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [ ]:
def create_entity_node(tx, entity):
    label = safe_label(entity["type"])

    props = {
        "id": entity["id"],
        "kg_type": entity["type"],
        "label": entity.get("label"),
        "confidence": entity.get("confidence"),
    }

    props.update(entity.get("properties", {}))

    evidence = entity.get("evidence_text", [])
    if isinstance(evidence, str):
        evidence = [evidence]

    props["evidence_text"] = evidence

    props = clean_props(props)

    query = f"""
    MERGE (n:{label} {{id: $id}})
    SET n += $props
    """

    tx.run(query, id=entity["id"], props=props)


with driver.session() as session:
    for entity in entities:
        session.execute_write(create_entity_node, entity)

print("Entity nodes loaded:", len(entities))

In [ ]:
entities_by_id = {e["id"]: e for e in entities}

party_by_canonical = {}
definedterm_by_normalized_term = {}
value_by_id = {}

for e in entities:
    props = e.get("properties", {})

    if e["type"] == "Party":
        canonical = props.get("canonical_name") or props.get("name")
        canonical = normalize_text(canonical)
        if canonical:
            party_by_canonical[canonical] = e["id"]

    elif e["type"] == "DefinedTerm":
        term = props.get("term") or e.get("label")
        term_norm = normalize_text(term)
        if term_norm:
            definedterm_by_normalized_term[term_norm] = e["id"]

    elif e["type"] == "Value":
        value_by_id[e["id"]] = e["id"]

print("Parties:", party_by_canonical)
print("DefinedTerms:", len(definedterm_by_normalized_term))
print("Values:", len(value_by_id))

In [ ]:
ALLOWED_REL_TYPES = {
    "IS_PART_OF",
    "CONTAINS",
    "REFERENCES",
    "DEFINES",
    "USES",
    "ASSIGNS_OBLIGATION_TO",
    "GRANTS_RIGHT_TO",
    "DEPENDS_ON",
    "MODIFIES_AMENDS",
    "SUPERSEDES",
}

def safe_rel_type(rel_type):
    if rel_type not in ALLOWED_REL_TYPES:
        raise ValueError(f"Invalid relationship type: {rel_type}")
    return rel_type


def create_explicit_relation(tx, rel):
    source = rel.get("source")
    target = rel.get("target")

    if not source or not target:
        return

    if source not in entities_by_id or target not in entities_by_id:
        return

    rel_type = safe_rel_type(rel["type"])

    props = {
        "confidence": rel.get("confidence"),
        "evidence_text": rel.get("evidence_text"),
        "source_field": rel.get("source_field"),
        "target_field": rel.get("target_field"),
    }

    props = clean_props(props)

    query = f"""
    MATCH (a {{id: $source}})
    MATCH (b {{id: $target}})
    MERGE (a)-[r:{rel_type}]->(b)
    SET r += $props
    """

    tx.run(query, source=source, target=target, props=props)


with driver.session() as session:
    for rel in relations:
        session.execute_write(create_explicit_relation, rel)

print("Explicit relations loaded:", len(relations))

In [ ]:
EVENT_TYPES = {"Obligation", "Right", "Permission", "Prohibition"}

def get_actor_field(entity):
    props = entity.get("properties", {})

    if entity["type"] == "Obligation":
        return props.get("actor")

    if entity["type"] in {"Right", "Permission"}:
        return props.get("holder") or props.get("actor")

    if entity["type"] == "Prohibition":
        return props.get("subject") or props.get("actor")

    return None


def create_event_party_relation(tx, event_id, event_type, party_id):
    if event_type == "Obligation":
        rel_type = "ASSIGNED_TO"
    elif event_type in {"Right", "Permission"}:
        rel_type = "GRANTED_TO"
    elif event_type == "Prohibition":
        rel_type = "RESTRICTS"
    else:
        rel_type = "RELATED_TO"

    query = f"""
    MATCH (e {{id: $event_id}})
    MATCH (p:Party {{id: $party_id}})
    MERGE (e)-[:{rel_type}]->(p)
    """

    tx.run(query, event_id=event_id, party_id=party_id)


def create_event_object_relation(tx, event_id, object_id):
    query = """
    MATCH (e {id: $event_id})
    MATCH (o {id: $object_id})
    MERGE (e)-[:HAS_OBJECT]->(o)
    """

    tx.run(query, event_id=event_id, object_id=object_id)


def create_event_value_relation(tx, event_id, value_id):
    query = """
    MATCH (e {id: $event_id})
    MATCH (v:Value {id: $value_id})
    MERGE (e)-[:HAS_VALUE]->(v)
    """

    tx.run(query, event_id=event_id, value_id=value_id)

In [ ]:
semantic_rel_count = 0

with driver.session() as session:
    for entity in entities:
        if entity["type"] not in EVENT_TYPES:
            continue

        props = entity.get("properties", {})
        event_id = entity["id"]

        actor = get_actor_field(entity)
        actor_norm = normalize_text(actor)

        if actor_norm in party_by_canonical:
            session.execute_write(
                create_event_party_relation,
                event_id,
                entity["type"],
                party_by_canonical[actor_norm],
            )
            semantic_rel_count += 1

        obj = props.get("normalized_object") or props.get("object")
        obj_norm = normalize_text(obj)

        if obj_norm in definedterm_by_normalized_term:
            session.execute_write(
                create_event_object_relation,
                event_id,
                definedterm_by_normalized_term[obj_norm],
            )
            semantic_rel_count += 1

        for value_id in props.get("value_refs", []):
            value_id_norm = normalize_text(value_id)

            matched_value_id = None
            if value_id in value_by_id:
                matched_value_id = value_id
            elif value_id_norm in value_by_id:
                matched_value_id = value_id_norm

            if matched_value_id:
                session.execute_write(
                    create_event_value_relation,
                    event_id,
                    matched_value_id,
                )
                semantic_rel_count += 1

print("Semantic relations created:", semantic_rel_count)

In [ ]:
def create_action_relation(tx, event_id, action_name):
    action_id = f"action_{normalize_text(action_name)}"

    query = """
    MERGE (a:Action {id: $action_id})
    SET a.name = $action_name

    WITH a
    MATCH (e {id: $event_id})
    MERGE (e)-[:HAS_ACTION]->(a)
    """

    tx.run(
        query,
        event_id=event_id,
        action_id=action_id,
        action_name=action_name,
    )


action_count = 0

with driver.session() as session:
    for entity in entities:
        if entity["type"] not in EVENT_TYPES:
            continue

        props = entity.get("properties", {})
        action = props.get("normalized_action") or props.get("action")

        if action:
            session.execute_write(
                create_action_relation,
                entity["id"],
                action,
            )
            action_count += 1

print("Action relations created:", action_count)

In [ ]:
clause_entities = [e for e in entities if e["type"] == "Clause"]
event_entities = [e for e in entities if e["type"] in EVENT_TYPES]

def create_clause_event_relation(tx, clause_id, event_id):
    query = """
    MATCH (c:Clause {id: $clause_id})
    MATCH (e {id: $event_id})
    MERGE (c)-[:CONTAINS_EVENT]->(e)
    """

    tx.run(query, clause_id=clause_id, event_id=event_id)


def evidence_matches_clause(event, clause):
    clause_text = clause.get("properties", {}).get("text", "")
    evidence = event.get("evidence_text", [])

    if isinstance(evidence, str):
        evidence = [evidence]

    clause_text_norm = clause_text.lower()

    for ev in evidence:
        ev_norm = str(ev).lower()

        if len(ev_norm) > 40 and ev_norm[:80] in clause_text_norm:
            return True

        if len(clause_text_norm) > 40 and clause_text_norm[:80] in ev_norm:
            return True

    return False


clause_event_count = 0

with driver.session() as session:
    for event in event_entities:
        for clause in clause_entities:
            if evidence_matches_clause(event, clause):
                session.execute_write(
                    create_clause_event_relation,
                    clause["id"],
                    event["id"],
                )
                clause_event_count += 1
                break

print("Clause-event relations created:", clause_event_count)

In [ ]:
def count_nodes_and_rels(tx):
    node_count = tx.run("MATCH (n) RETURN count(n) AS count").single()["count"]
    rel_count = tx.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]

    labels = tx.run("""
    MATCH (n)
    RETURN labels(n) AS labels, count(*) AS count
    ORDER BY count DESC
    """).data()

    rel_types = tx.run("""
    MATCH ()-[r]->()
    RETURN type(r) AS type, count(*) AS count
    ORDER BY count DESC
    """).data()

    return node_count, rel_count, labels, rel_types


with driver.session() as session:
    node_count, rel_count, labels, rel_types = session.execute_read(count_nodes_and_rels)

print("Nodes:", node_count)
print("Relationships:", rel_count)

print("\nLabels:")
for row in labels:
    print(row)

print("\nRelationship types:")
for row in rel_types:
    print(row)

In [ ]:
def get_events_by_party(tx, party_name):
    query = """
    MATCH (e)-[r]->(p:Party)
    WHERE p.canonical_name = $party_name
    RETURN e.id AS event_id, labels(e) AS labels, e.label AS label, type(r) AS rel_type
    LIMIT 25
    """

    return tx.run(query, party_name=party_name).data()


with driver.session() as session:
    rows = session.execute_read(get_events_by_party, "bellicum")

for row in rows:
    print(row)

In [ ]:
def get_candidate_event_pairs(tx):
    query = """
    MATCH (e1)-[:HAS_ACTION]->(a:Action)<-[:HAS_ACTION]-(e2)
    WHERE id(e1) < id(e2)

    OPTIONAL MATCH (e1)-[:HAS_OBJECT]->(o)<-[:HAS_OBJECT]-(e2)

    RETURN 
        e1.id AS event_1,
        e1.label AS label_1,
        e2.id AS event_2,
        e2.label AS label_2,
        a.name AS shared_action,
        o.label AS shared_object
    LIMIT 50
    """

    return tx.run(query).data()


with driver.session() as session:
    rows = session.execute_read(get_candidate_event_pairs)

for row in rows[:10]:
    print(json.dumps(row, indent=2, ensure_ascii=False))

In [ ]:
# driver.close()
# print("Neo4j connection closed.")